In [ ]:
import os
from typing import Optional
import nemo_run as run

def slurm_executor(
    user: str,
    host: str,
    identity_file: str,
    remote_job_dir: str,
    account: str,
    partition: str,
    nodes: int,
    devices: int,
    time: str = "01:00:00",
    custom_env_vars: Optional[dict[str, str]] = None,
    retries: int = 0,
) -> run.SlurmExecutor:
    if not (user and host and remote_job_dir and account and partition and nodes and devices):
        raise RuntimeError(
            "Please set user, host, remote_job_dir, account, partition, nodes, and devices args for using this function."
        )

    # Env vars for jobs are configured here
    env_vars = {
        "TORCH_NCCL_AVOID_RECORD_STREAMS": "1",
        "NCCL_NVLS_ENABLE": "0",
        "NVTE_DP_AMAX_REDUCE_INTERVAL": "0",
        "NVTE_ASYNC_AMAX_REDUCTION": "1",
    }
    if custom_env_vars:
        env_vars |= custom_env_vars

    # This will package the train.py script in the current working directory to the remote cluster.
    # If you are inside a git repo, you can also use https://github.com/NVIDIA/NeMo-Run/blob/main/src/nemo_run/core/packaging/git.py.
    # If the script already exists on your container and you call it with the absolute path, you can also just use `run.Packager()`.
    packager = run.PatternPackager(include_pattern="train_dev.py", relative_path=os.getcwd())

    # This defines the slurm executor.
    # We connect to the executor via the tunnel defined by user, host and remote_job_dir.
    executor = run.SlurmExecutor(
        account=account,
        partition=partition,
        tunnel=run.SSHTunnel(
            user=user,
            host=host,
            job_dir=remote_job_dir, # This is where the results of the run will be stored by default.
            identity=identity_file # OPTIONAL: Provide path to the private key that can be used to establish the SSH connection without entering your password.
        ),
        nodes=nodes,
        ntasks_per_node=devices,
        gpus_per_node=devices,
        exclusive=True,
        # gres="gpu:4",
        packager=packager,
    )

    executor.env_vars = env_vars
    executor.retries = retries
    executor.time = time
    return executor


training_job = run.Script(
    inline="""
python train_dev.py
""",
    entrypoint = """singularity exec --nv \
--bind /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2.0_training_experiment/nemo_2.0_training_experiment_1744150057/training:/nemo_run \ #dev !!!! ale ze mnie debil
--pwd /nemo_run/code \
/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/images/nemo_dev.sif \
cat"""
)

# Run it locally
# executor = run.LocalExecutor()
executor = slurm_executor(
    user="plgmstefaniak",
    host="helios.cyfronet.pl",
    identity_file="/home/maciej/.ssh/plgrid",
    remote_job_dir="/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary",
    # remote_job_dir="net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/llm_random_cemetery",
    # remote_job_dir="/net/home/plgrid/plgmstefaniak/tmp/",
    account="plgllmefficont2-gpu-gh200",
    partition="plgrid-gpu-gh200",
    nodes=1,
    devices=4,

) # pass in args relevant to your cluster

with run.Experiment("nemo_2.0_training_experiment", log_level="INFO") as exp:
    exp.add(training_job, executor=executor, tail_logs=True, name="training")
    # Add more jobs as needed

    # Run the experiment
    exp.run(detach=False)




──────── Entering Experiment nemo_2.0_training_experiment with id: nemo_2.0_training_experiment_1744158261 ────────

[02:24:21] Connecting to plgmstefaniak@helios.cyfronet.pl                                             ]8;id=898073;file:///home/maciej/projects/cluster/nemorand/nemo_run/core/tunnel/client.py\client.py]8;;\:]8;id=629911;file:///home/maciej/projects/cluster/nemorand/nemo_run/core/tunnel/client.py#257\257]8;;\

Connected (version 2.0, client OpenSSH_8.0)
Authentication (publickey) successful!
rsyncing /home/maciej/.nemo_run/experiments/nemo_2.0_training_experiment/nemo_2.0_training_experiment_1744158261 to /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2.0_training_experiment ...
Successfully ran `rsync  -pthrvz  --rsh='ssh -i /home/maciej/.ssh/plgrid -p 22 ' /home/maciej/.nemo_run/experiments/nemo_2.0_training_experiment/nemo_2.0_training_experiment_1744158261 plgmstefaniak@helios.cyfronet.pl:/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2.0_training_experiment`


[02:24:25] Launching job training for experiment nemo_2.0_training_experiment                     ]8;id=508962;file:///home/maciej/projects/cluster/nemorand/nemo_run/run/experiment.py\experiment.py]8;;\:]8;id=458577;file:///home/maciej/projects/cluster/nemorand/nemo_run/run/experiment.py#744\744]8;;\

Launched app: slurm_tunnel://nemo_run/144045


...

Waiting for job 144045 to finish [log=True]...


[02:24:31] Waiting for app state response before fetching logs...                                       ]8;id=286871;file:///home/maciej/projects/cluster/nemorand/nemo_run/run/logs.py\logs.py]8;;\:]8;id=544706;file:///home/maciej/projects/cluster/nemorand/nemo_run/run/logs.py#105\105]8;;\

Job 144045 finished: SUCCEEDED


                                                                                                                   
# The experiment was run with the following tasks: ['training']                                                    
# You can inspect and reconstruct this experiment at a later point in time using:                                  
experiment = run.Experiment.from_id("nemo_2.0_training_experiment_1744158261")                                     
experiment.status() # Gets the overall status                                                                      
experiment.logs("training") # Gets the log for the provided task                                                   
experiment.cancel("training") # Cancels the provided task if still running                                         
                                                                                                                   

                                                                                                                   
# You can inspect this experiment at a later point in time using the CLI as well:                                  
nemo experiment status nemo_2.0_training_experiment_1744158261                                                     
nemo experiment logs nemo_2.0_training_experiment_1744158261 0                                                     
nemo experiment cancel nemo_2.0_training_experiment_1744158261 0                                                   
                                                                                                                   

In [2]:
# srun 
# --output /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2.0_training_experiment/nemo_2.0_training_experiment_1744148528/training/log-plgllmefficont2-gpu-gh200-plgllmefficont2-gpu-gh200.training_%j_${SLURM_RESTART_COUNT:-0}.out 
# --container-mounts /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2.0_training_experiment/nemo_2.0_training_experiment_1744148528/training:/nemo_run 
# --container-workdir /nemo_run/code 
# --wait=60 --kill-on-bad-exit=1 bash /nemo_run/scripts/training.sh

In [3]:
import nemo_run as run

experiment = run.Experiment.from_id("nemo_2.0_training_experiment_1744157432")                                     
experiment.status() # Gets the overall status                                                                      
experiment.logs("training") # Gets the log for the provided task                                                   
# experiment.cancel("training") # Cancels the provided task if still running

[02:24:37] Connecting to plgmstefaniak@helios.cyfronet.pl                                             ]8;id=883777;file:///home/maciej/projects/cluster/nemorand/nemo_run/core/tunnel/client.py\client.py]8;;\:]8;id=527325;file:///home/maciej/projects/cluster/nemorand/nemo_run/core/tunnel/client.py#257\257]8;;\

Connected (version 2.0, client OpenSSH_8.0)
Authentication (publickey) successful!
[chan 13] Opened sftp connection (server version 3)


training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 15:4: not a valid test operator:  
training/0 15:4: not a valid test operator: 12.8
training/0 21:4: not a valid test operator: (
training/0 21:4: not a valid test operator: 565.57.01
training/0 15:4: not a valid test operator:  
training/0 15:4: not a valid test operator: 12.8
training/0 21:

Experiment Status for nemo_2.0_training_experiment_1744157432

Task 0: training
- Status: SUCCEEDED
- Executor: SlurmExecutor on plgmstefaniak@helios.cyfronet.pl
- Job id: 144044
- Local Directory: /home/maciej/.nemo_run/experiments/nemo_2.0_training_experiment/nemo_2.0_training_experiment_1744157432/training
- Remote Directory: /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2.0_training_experiment/nemo_2.0_training_experiment_1744157432/training

[02:24:41] Fetching logs for training                                                             ]8;id=287159;file:///home/maciej/projects/cluster/nemorand/nemo_run/run/experiment.py\experiment.py]8;;\:]8;id=198273;file:///home/maciej/projects/cluster/nemorand/nemo_run/run/experiment.py#931\931]8;;\

[chan 7] Opened sftp connection (server version 3)
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 15:4: not a valid test operator:  
training/0 15:4: not a valid test operator: 12.8
training/0 21:4: not a valid test operator: (
training/0 21:4: not a valid test operator: 565.57.01
training/0 15:4: not a valid test operator:  
training/0 1

In [4]:
experiment

Graphviz rendering failed: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH
